In [1]:
import pandas as pd
from autogluon.tabular import TabularPredictor

# 1. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 피처 엔지니어링 함수 정의
def engineer_features(df):
    # (1) 순서형 데이터 수치화
    mapping = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}
    count_cols = [
        '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
        '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
        '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
    ]
    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(-1)

    # (2) 나이대 수치화
    age_map = {
        '만18-34세': 26, '만35-37세': 36, '만38-39세': 38.5,
        '만40-42세': 41, '만43-44세': 43.5, '만45-50세': 43.5, '알 수 없음': 43.5
    }
    if '시술 당시 나이' in df.columns:
        df['시술_당시_나이_수치'] = df['시술 당시 나이'].map(age_map)
    
    # (3) 시험관/전환율 관련
    df['난자→배아_전환율'] = df['총 생성 배아 수'].fillna(0) / (df['수집된 신선 난자 수'].fillna(0) + 1)
    df['수정_성공률'] = df['혼합된 난자 수'].fillna(0) / (df['수집된 신선 난자 수'].fillna(0) + 1)
    df['배아_동결_비율'] = df['저장된 배아 수'].fillna(0) / (df['총 생성 배아 수'].fillna(0) + 1)
    df['이식_성공_비율'] = df['이식된 배아 수'].fillna(0) / (df['총 생성 배아 수'].fillna(0) + 1) 
    df['포배기_이식_여부'] = (df['배아 이식 경과일'] == 5.0).astype(int)
    df['분할기_이식_여부'] = (df['배아 이식 경과일'] == 3.0).astype(int)

    # (4) 과거력 및 효율
    df['과거_시술_성공_효율'] = df['총 출산 횟수'] / (df['총 시술 횟수'] + 1)
    df['순수_실패_횟수'] = df['총 시술 횟수'] - df['총 임신 횟수']
    
    return df

# 3. 데이터 적용 (먼저 변수를 생성함)
train_processed = engineer_features(train)
test_processed = engineer_features(test)

# 4. 불필요한 피처 및 중복 피처 삭제 (학습 직전 단계)
drop_features = [
    '시술 유형', 
    '불임 원인 - 자궁경부 문제', 
    '불임 원인 - 정자 농도', 
    '불임 원인 - 정자 형태', 
    '불임 원인 - 정자 운동성',
    '대리모 여부', 
    '난자 해동 경과일', 
    '저장된 신선 난자 수',
    'DI 출산 횟수',
    'DI 임신 횟수',
    '시술 당시 나이',  # 수치형 변수를 만들었으므로 원본 문자열은 삭제
    'ID'              # ID는 학습에 불필요
]

# 실제로 데이터에 존재하는 컬럼만 골라서 삭제
final_train_data = train_processed.drop(columns=[col for col in drop_features if col in train_processed.columns])
final_test_data = test_processed.drop(columns=[col for col in drop_features if col in test_processed.columns])

# 5. AutoGluon 학습
predictor = TabularPredictor(
    label='임신 성공 여부', 
    eval_metric='roc_auc'
).fit(
    final_train_data,
    presets='best_quality',
    time_limit=7200,
    num_stack_levels=1
)

# 6. 리더보드 확인
leaderboard = predictor.leaderboard(final_train_data, silent=True)
print(leaderboard)

No path specified. Models will be saved in: "AutogluonModels/ag-20260210_082536"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.11
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.2.0: Tue Nov 18 21:09:40 PST 2025; root:xnu-12377.61.12~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       1.84 GB / 16.00 GB (11.5%)
Disk Space Avail:   297.39 GB / 460.43 GB (64.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyS

                          model  score_test  score_val eval_metric  \
0          LightGBMLarge_BAG_L1    0.771218   0.738811     roc_auc   
1          LightGBM_r131_BAG_L1    0.757729   0.739862     roc_auc   
2          LightGBMLarge_BAG_L2    0.755935   0.739748     roc_auc   
3             LightGBMXT_BAG_L2    0.753854   0.740348     roc_auc   
4          LightGBM_r131_BAG_L2    0.753829   0.740415     roc_auc   
5                XGBoost_BAG_L1    0.753791   0.739101     roc_auc   
6          CatBoost_r177_BAG_L2    0.753621   0.740445     roc_auc   
7               CatBoost_BAG_L2    0.753109   0.740451     roc_auc   
8               LightGBM_BAG_L2    0.752828   0.740371     roc_auc   
9                XGBoost_BAG_L2    0.752787   0.740215     roc_auc   
10        NeuralNetTorch_BAG_L2    0.752130   0.739561     roc_auc   
11         CatBoost_r177_BAG_L1    0.752036   0.739990     roc_auc   
12    NeuralNetTorch_r79_BAG_L2    0.751872   0.739989     roc_auc   
13              Ligh

In [3]:
predictions_proba = predictor.predict_proba(final_test_data)

# 임신 성공(1)에 해당하는 확률만 추출
final_preds = predictions_proba.iloc[:, 1] 
 
# 7. 제출 파일 생성
submission = pd.DataFrame({
    'ID': test['ID'],  # 원본 test의 ID를 그대로 사용
    'probability': final_preds
})

# 파일 저장
submission.to_csv('feature_5_2hr_drop_autogluon_submission.csv', index=False, encoding='utf-8')
print("--- 제출 파일 생성 완료 ---")

--- 제출 파일 생성 완료 ---
